In [1]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)} | "
            f"VRAM: {props.total_memory / 1024**3:.2f} GB"
        )

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | VRAM: 14.56 GB
GPU 1: Tesla T4 | VRAM: 14.56 GB


In [2]:
import requests

response = requests.get(
    "https://huggingface.co",
    timeout=10
)

print("Internet status:", response.status_code)

Internet status: 200


In [3]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)

/kaggle/input/datasets/hassanch6138/localsql-phase3-input/manifest.jsonl
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/.gitignore
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/pyproject.toml
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/README.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/uv.lock
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/.python-version
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/CLAUDE.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/tests/__init__.py
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/localsql-phase3-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src/localsql-phase3-s

In [4]:
from pathlib import Path
import shutil

SOURCE_DIR = Path(
    "/kaggle/input/datasets/hassanch6138/localsql-phase3-input/localsql-phase3-src"
)

MANIFEST_SOURCE = Path(
    "/kaggle/input/datasets/hassanch6138/localsql-phase3-input/manifest.jsonl"
)

PROJECT_DIR = Path("/kaggle/working/localsql")

# Clean old working copy if this cell is rerun
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

# Copy source, but ignore the accidental nested duplicate repo
shutil.copytree(
    SOURCE_DIR,
    PROJECT_DIR,
    ignore=shutil.ignore_patterns("localsql-phase3-src")
)

print("Project copied to:", PROJECT_DIR)
print("Manifest source:", MANIFEST_SOURCE)

print("\nTop-level project files:")
for p in sorted(PROJECT_DIR.iterdir()):
    print(" -", p.name)

Project copied to: /kaggle/working/localsql
Manifest source: /kaggle/input/datasets/hassanch6138/localsql-phase3-input/manifest.jsonl

Top-level project files:
 - .gitignore
 - .python-version
 - CLAUDE.md
 - PROJECT.md
 - README.md
 - configs
 - data
 - docs
 - pyproject.toml
 - scripts
 - src
 - tests
 - uv.lock


In [5]:
MANIFEST_DEST = (
    PROJECT_DIR
    / "data"
    / "benchmarks"
    / "bird_mini_dev"
    / "generation"
    / "manifest.jsonl"
)

MANIFEST_DEST.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(MANIFEST_SOURCE, MANIFEST_DEST)

print("Manifest copied to:")
print(MANIFEST_DEST)
print("Exists:", MANIFEST_DEST.exists())
print("Size:", MANIFEST_DEST.stat().st_size, "bytes")

Manifest copied to:
/kaggle/working/localsql/data/benchmarks/bird_mini_dev/generation/manifest.jsonl
Exists: True
Size: 4223357 bytes


In [6]:
import json

count = 0
first_example = None

with open(MANIFEST_DEST, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)

            if first_example is None:
                first_example = row

            count += 1

print("Total examples:", count)
print("First example ID:", first_example["example_id"])
print("First database:", first_example["db_id"])
print("\nFirst question:")
print(first_example["question"])

Total examples: 500
First example ID: bird-mini-dev-sqlite-0000
First database: debit_card_specializing

First question:
What is the ratio of customers who pay in EUR against customers who pay in CZK?


In [7]:
forbidden_fields = {
    "sql",
    "gold_sql",
    "target_sql",
    "completion",
    "reference_sql",
}

found_forbidden = forbidden_fields.intersection(first_example.keys())

print("Fields in first example:")
print(sorted(first_example.keys()))

assert not found_forbidden, (
    f"Gold leakage detected: {found_forbidden}"
)

print("\nGold leakage check: PASSED")

Fields in first example:
['business_context', 'db_id', 'dialect', 'difficulty', 'example_id', 'prompt', 'question', 'serialized_schema']

Gold leakage check: PASSED


In [8]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [9]:
!pwd
!ls

/kaggle/working/localsql
CLAUDE.md  data  PROJECT.md	 README.md  src    uv.lock
configs    docs  pyproject.toml  scripts    tests


In [10]:
import importlib.util
import torch

packages = [
    "transformers",
    "accelerate",
    "bitsandbytes",
]

print("Torch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

for pkg in packages:
    print(
        f"{pkg}:",
        "installed"
        if importlib.util.find_spec(pkg)
        else "NOT installed"
    )

Torch: 2.10.0+cu128
CUDA runtime: 12.8
CUDA available: True
transformers: installed
accelerate: installed
bitsandbytes: NOT installed


In [11]:
!pip install -q -e .
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for localsql (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 93.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.1 MB/s eta 0:00:00:00:01


In [12]:
import torch
import transformers
import accelerate
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

torch: 2.10.0+cu128
transformers: 5.17.0
accelerate: 1.15.0
bitsandbytes: 0.50.2
CUDA: 12.8
CUDA available: True


In [13]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print("Using physical GPU:", os.environ["CUDA_VISIBLE_DEVICES"])

Using physical GPU: 0


In [14]:
!python scripts/run_baseline.py \
    --manifest data/benchmarks/bird_mini_dev/generation/manifest.jsonl \
    --run-id kaggle-dry-run \
    --dry-run \
    --limit 5

Manifest: 5 gold-free examples parsed OK (no gold fields present).
Manifest sha256: 6903e95f373e267d5ebbbf5fa9b47100c3cf3ded9294595e1c5a026dbe0befa7
Run directory: /kaggle/working/localsql/data/runs/kaggle-dry-run
Already completed (resume): 0 / 5
Remaining to generate: 5
Dry run OK -- no model loaded, no CUDA required.
